# Section 2: 멀티소스 팩트체커

뉴스 기사나 SNS 주장의 진위를 **여러 소스에서 교차 검증**하고,
이전 검증 결과를 기억하며 후속 질문에 답변하는 에이전트를 구축합니다.

### 학습 목표
- @tool 데코레이터로 커스텀 Tool 제작
- create_agent로 Agent 생성
- RunnableWithMessageHistory로 대화 맥락 유지
- 여러 소스의 결과를 종합하여 판정

## 0. 패키지 설치 및 환경 설정

## 1. 검증 도구(Tool) 제작

팩트체커가 사용할 3가지 검색 도구를 만듭니다.

## 2. Agent 생성

3가지 도구를 가진 **ReAct 에이전트**를 만듭니다.

## 3. Memory 추가 — 대화 맥락 유지

이전 검증 결과를 기억하고 후속 질문에 활용할 수 있도록 합니다.

## 4. 배치 검증 — 여러 주장을 한 번에 검증

## 5. 스트리밍 — 실시간으로 검증 과정 확인

## 6. 비용·반복 가드 (Cost & Loop Guards)

### 왜 중요한가

앞서 만든 `fact_checker_agent`는 ReAct 패턴으로 **모델이 스스로 tool 호출 횟수를 결정**합니다.
시스템 프롬프트에 "각 1회씩만 호출"이라고 자연어로 지시했지만, 이는 **보장이 아니라 권고**입니다.
모델이 판단을 바꿔 동일 도구를 반복 호출하거나 tool 실패 후 재시도하면 **1회 실행에 10~20회 LLM 호출**이 누적될 수 있습니다.

### 3가지 가드

| 가드 | 어디에 | 역할 |
|------|--------|------|
| `max_tokens` | `ChatUpstage(...)` | 단일 호출 **출력 토큰 상한** → 응답 길이 폭주 차단 |
| `timeout` / `max_retries` | `ChatUpstage(...)` | 네트워크 장애 시 **무한 대기·재시도 차단** |
| `recursion_limit` | `agent.invoke(config=...)` | 에이전트의 **tool 호출 hop 수 상한** → 루프 차단 |

`create_agent`는 내부적으로 LangGraph 위에 구축되어, `config`의 `recursion_limit`이
hop 상한으로 동작합니다(미설정 시 기본값 25). 학습 환경에서는 **10 이하로 제한**하는 것을 권장합니다.

### 실패 시 동작
`recursion_limit` 초과 시 `GraphRecursionError`가 발생합니다. 이는 **버그 조기 발견 신호**이므로
에러를 삼키지 말고, 프롬프트·도구·그래프 설계를 점검하는 트리거로 활용하세요.


## 정리 및 핵심 요약

### 오늘 구축한 것

```
사용자 주장
    |
    v
[Agent (ReAct)] -- 판단 --> [웹 검색 Tool]
    |                   --> [위키피디아 Tool]
    |                   --> [계산 Tool]
    |
    v
[다중 소스 교차 검증]
    |
    v
[판정 리포트 생성]
    |
    v
[Memory] -- 이전 검증 기억 --> 후속 질문 처리
```

### 새로 배운 개념

| 개념 | 역할 |
|------|------|
| @tool | 함수를 Agent 도구로 정의 |
| create_agent | ReAct 패턴 Agent 생성 |
| MemorySaver | 대화 기록 유지 (체크포인터) |
| thread_id | 세션별 대화 분리 |
| stream_mode="updates" | 실시간 동작 추적 |

### 다음 시간: 자율과제

Agent + Tool + Memory를 활용하여 다른 도메인의 에이전트를 만들어보세요!

## ⚠️ ReAct 폭주 가드 — `recursion_limit` + 도구 호출 카운터

### 무엇이 문제인가

이 노트북의 에이전트는 **ReAct 패턴**(또는 LangGraph supervisor 루프)으로 동작합니다.
LLM이 스스로 "어떤 도구를 부를지 / 언제 멈출지"를 결정하므로, 한 번의 `invoke`가
실제로는 **3~10회 LLM 호출**로 펼쳐집니다.

모델이 같은 도구를 반복 호출하거나 결론을 못 내려 루프를 돌면, 한 번 실행에
**10~20회 이상** API가 호출되어 다음과 같은 사고가 납니다.

- 💸 비용 폭주
- 🚫 Upstage tier 0 같은 무료 한도에서 `429 RateLimitError`
- ⏱️ 응답 시간 폭증

### 두 가지 가드

#### 1️⃣ `recursion_limit` — 전체 hop 상한

LangGraph가 한 번의 `invoke` 안에서 노드 전이를 몇 번까지 허용할지 제한합니다.
초과하면 `GraphRecursionError`로 **즉시 멈추므로** 폭주가 결정적으로 차단됩니다.

```python
agent.invoke({"messages": [...]}, config={"recursion_limit": 25})
```

> 🛗 **비유**: 엘리베이터가 8층 이상 못 올라가도록 막아둔 안전장치.

#### 2️⃣ 도구 호출 카운터 — 도구별 N회 상한

ReAct 폭주의 진짜 원인은 **"같은 도구를 반복 호출"** 입니다.
도구를 카운터로 감싸서 N회 초과 시 *"이미 N회 불렀으니 결론을 내세요"* 메시지를
강제로 돌려주면, ReAct는 자연스럽게 답변 단계로 넘어갑니다.

```python
tools = with_call_quota([search_web, search_wikipedia, calculate], limit=2)
```

> 🧻 **비유**: 휴지가 2칸씩만 끊어 나오는 디스펜서. 더 뽑으려 해도 멈춰 있습니다.

### 함께 쓰면 어떤 폭주든 둘 중 하나에 걸린다

| 가드 | 역할 | 한도 초과 시 |
|------|------|-------------|
| `recursion_limit` | 전체 hop 수 상한 | `GraphRecursionError` 발생 → 강제 종료 |
| 도구 호출 카운터 | 도구별 호출 횟수 상한 | quota 메시지 반환 → 모델이 결론으로 진행 |

### 이 노트북에서의 사용

아래 셀에서 두 헬퍼를 정의합니다.

- `with_call_quota(tools)` : 도구 리스트의 각 도구에 호출 한도 적용
- `with_guards(config)`   : `invoke`/`stream`에 `recursion_limit` 자동 주입

이후 모든 agent/graph 호출에 두 가드가 자동 적용됩니다.
